# Latest built

In [6]:
import cv2
import mediapipe as mp
import pyautogui
import time
import random
import math
import numpy as np
import os

In [8]:
def create_shape():
    size = random.randint(30, 60)
    x = random.randint(size, width - size)
    y = 0
    vy = random.uniform(2, 5)
    if random.random() > 0.3 and loaded_images:
        img_idx = random.randint(0, len(loaded_images) - 1)
        return [x, y, size, "image", img_idx, None, vy]
    else:
        shape_type = random.choice(list(color_options.keys()))
        color, color_name = color_options[shape_type]
        return [x, y, size, shape_type, color_name, color, vy]

def draw_triangle(img, x, y, size, color):
    pt1 = (int(x), int(y - size))
    pt2 = (int(x - size), int(y + size))
    pt3 = (int(x + size), int(y + size))
    pts = [pt1, pt2, pt3]
    cv2.drawContours(img, [np.array(pts)], 0, color, -1)

In [10]:
def overlay_image(bg, img, x, y, target_width):
    orig_h, orig_w = img.shape[:2]
    aspect_ratio = orig_h / orig_w
    target_height = int(target_width * aspect_ratio)
    img_resized = cv2.resize(img, (target_width, target_height))

    x1 = max(x, 0)
    y1 = max(y, 0)
    x2 = min(x + target_width, bg.shape[1])
    y2 = min(y + target_height, bg.shape[0])

    img_w = x2 - x1
    img_h = y2 - y1
    if img_w <= 0 or img_h <= 0:
        return

    img_cropped = img_resized[0:img_h, 0:img_w]

    # If image has alpha channel (4 channels)
    if img_cropped.shape[2] == 4:
        alpha_s = img_cropped[:, :, 3] / 255.0
        alpha_l = 1.0 - alpha_s
        for c in range(3):
            bg[y1:y2, x1:x2, c] = (alpha_s * img_cropped[:, :, c] + 
                                   alpha_l * bg[y1:y2, x1:x2, c])
    else:
        # Direct paste for 3-channel image
        bg[y1:y2, x1:x2] = img_cropped

In [12]:
# MediaPipe setup
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils

# Shape and score setup
shapes = []  # [x, y, size, shape_type, color_name/img_idx, color, vy]
shape_interval = random.randrange(1, 3)
last_shape_time = time.time()
OVER = 0

color_options = {
    "circle": ((0, 0, 255), "red"),
    "square": ((0, 255, 0), "green"),
    "triangle": ((255, 0, 0), "blue")
}
color_scores = {"red": 0, "green": 0, "blue": 0}

# Load images
image_folder = "images"
image_paths = [os.path.join(image_folder, img) for img in os.listdir(image_folder) if img.endswith(('.png', '.PNG'))]
loaded_images = [cv2.imread(path, cv2.IMREAD_UNCHANGED) for path in image_paths if cv2.imread(path, cv2.IMREAD_UNCHANGED) is not None]

for i, img in enumerate(loaded_images):
    print(f"Image {i}: shape = {img.shape}, channels = {img.shape[2]}")

# Webcam
cap = cv2.VideoCapture('road1.mp4')
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

In [14]:
while cap.isOpened():
    current_time = time.time()
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)

    if OVER == 3:
        message = "Game Over"
        font = cv2.FONT_HERSHEY_COMPLEX
        font_scale = 2
        thickness = 3
        (text_width, text_height), _ = cv2.getTextSize(message, font, font_scale, thickness)
        text_x = (width - text_width) // 2
        text_y = (height + text_height) // 2
        cv2.rectangle(frame, (text_x - 20, text_y - text_height - 20),
                      (text_x + text_width + 20, text_y + 20), (255, 255, 255), -1)
        cv2.putText(frame, message, (text_x, text_y), font, font_scale, (0, 0, 0), thickness)
        cv2.imshow('Shape Buster Game (Punch to Pop)', frame)
        if cv2.waitKey(0) & 0xFF == ord('q'):
            break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb_frame)

    if current_time - last_shape_time > shape_interval:
        shapes.append(create_shape())
        last_shape_time = current_time

    updated_shapes = []
    for shape in shapes:
        x, y, size, shape_type, label, color, vy = shape
        y += vy
        if y - size > height:
            OVER += 1
            continue
        shape[1] = y

        if shape_type == "image":
            if 0 <= label < len(loaded_images):
                overlay_image(frame, loaded_images[label], int(x - size // 2), int(y - size // 2), size)
        elif shape_type == "circle":
            cv2.circle(frame, (int(x), int(y)), size, color, -1)
        elif shape_type == "square":
            cv2.rectangle(frame, (int(x - size), int(y - size)), (int(x + size), int(y + size)), color, -1)
        elif shape_type == "triangle":
            draw_triangle(frame, x, y, size, color)

        updated_shapes.append(shape)
    shapes = updated_shapes

    if result.multi_hand_landmarks:
        for landmarks in result.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, landmarks)
            h, w, _ = frame.shape
            index_tip = landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP]
            thumb_tip = landmarks.landmark[mp_hands.HandLandmark.THUMB_TIP]
            index_x, index_y = int(index_tip.x * w), int(index_tip.y * h)
            thumb_x, thumb_y = int(thumb_tip.x * w), int(thumb_tip.y * h)

            cv2.circle(frame, (index_x, index_y), 5, (0, 255, 255), -1)
            cv2.circle(frame, (thumb_x, thumb_y), 5, (255, 255, 0), -1)

            punch_distance = math.hypot(index_x - thumb_x, index_y - thumb_y)
            if punch_distance < 40:
                new_shapes = []
                for shape in shapes:
                    x, y, size, shape_type, label, color, vy = shape
                    distance_to_finger = math.hypot(x - index_x, y - index_y)
                    if distance_to_finger < size + 10:
                        if shape_type != "image":
                            color_scores[label] += 1
                            print(f"{label.capitalize()} {shape_type} busted! Score: {color_scores[label]}")
                        else:
                            print(f"Image object busted!")
                    else:
                        new_shapes.append(shape)
                shapes = new_shapes

    cv2.putText(frame, f"Red: {color_scores['red']}", (10, height - 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    cv2.putText(frame, f"Green: {color_scores['green']}", (10, height - 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.putText(frame, f"Blue: {color_scores['blue']}", (10, height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

    cv2.imshow('Shape Buster Game (Punch to Pop)', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# PyGame

In [3]:
import pygame
import random
import time
import os
import cv2
import mediapipe as mp

pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [4]:
# Load and display image
def overlay_image(bg, img, x, y, target_width):
    orig_h, orig_w = img.shape[:2]
    aspect_ratio = orig_h / orig_w
    target_height = int(target_width * aspect_ratio)
    img_resized = cv2.resize(img, (target_width, target_height))

    # Convert BGR to RGB for Pygame
    img_resized = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    img_surface = pygame.surfarray.make_surface(img_resized)
    bg.blit(img_surface, (x, y))

# Function to display the "Game Over" screen
def display_game_over():
    game_over_text = font.render("Game Over", True, (255, 0, 0))
    text_width, text_height = game_over_text.get_width(), game_over_text.get_height()
    text_x = (SCREEN_WIDTH - text_width) // 2
    text_y = ((SCREEN_HEIGHT - text_height) // 2) - 30
    pygame.draw.rect(screen, (255, 255, 255), 
                     (text_x - 20, text_y - 15, text_width + 40, text_height + 30))  # White background

    # Blit the "Game Over" text onto the screen
    screen.blit(game_over_text, (text_x, text_y))

    restart_text = font.render("Press 'R' to Restart or 'Q' to Quit", True, (0, 0, 0))
    restart_text_width, restart_text_height = restart_text.get_width(), restart_text.get_height()
    restart_text_x = (SCREEN_WIDTH - restart_text_width) // 2
    restart_text_y = (SCREEN_HEIGHT - restart_text_height) // 2 + 30
    pygame.draw.rect(screen, (255, 255, 255), 
                     (restart_text_x - 20, restart_text_y - 15, restart_text_width + 40, restart_text_height + 30))  # White background
    screen.blit(restart_text, (restart_text_x, restart_text_y))
    # pygame.display.update()    

In [5]:
# Define the Shape class
class Shape:
    def __init__(self, x, y, size, shape_type, color_name, color, vy, image_idx=None):
        self.x = x
        self.y = y
        self.size = size
        self.shape_type = shape_type
        self.color_name = color_name
        self.color = color
        self.vy = vy
        self.image_idx = image_idx

    def update(self):
        # Update the vertical position of the shape (falling)
        self.y += self.vy
        if self.y - self.size > SCREEN_HEIGHT:  # If it falls out of the screen, reset
            return False
        return True

    def draw(self, surface):
        if self.shape_type == "image" and self.image_idx is not None:
            overlay_image(surface, loaded_images[self.image_idx], self.x - self.size // 2, self.y - self.size // 2, self.size)
        elif self.shape_type == "circle":
            pygame.draw.circle(surface, self.color, (self.x, self.y), self.size)
        elif self.shape_type == "square":
            pygame.draw.rect(surface, self.color, (self.x - self.size, self.y - self.size, self.size * 2, self.size * 2))
        elif self.shape_type == "triangle":
            self.draw_triangle(surface)

    def draw_triangle(self, surface):
        pt1 = (self.x, self.y - self.size)
        pt2 = (self.x - self.size, self.y + self.size)
        pt3 = (self.x + self.size, self.y + self.size)
        pygame.draw.polygon(surface, self.color, [pt1, pt2, pt3])

In [35]:
# Game loop setup
cap = cv2.VideoCapture('road1.mp4')  # Use the webcam or video file
clock = pygame.time.Clock()

# Constants
SCREEN_WIDTH, SCREEN_HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = 30
SHAPE_FALL_INTERVAL_MIN = 1
SHAPE_FALL_INTERVAL_MAX = 2
MAX_SHAPES_DROPPED = 10
SHAPE_COLOR_SCHEME = {
    "circle": ((0, 0, 255), "red"),
    "square": ((0, 255, 0), "green"),
    "triangle": ((255, 0, 0), "blue")
}
IMAGE_FALL_PROBABILITY = 0.3

# Initialize Pygame
pygame.init()

# Set up the display
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Shape Buster Game")

# Font for text rendering
font = pygame.font.Font(None, 36)

# Load images (same as before)
image_folder = "images"
image_paths = [os.path.join(image_folder, img) for img in os.listdir(image_folder) if img.endswith(('.png', '.PNG'))]
loaded_images = [cv2.imread(path, cv2.IMREAD_UNCHANGED) for path in image_paths if cv2.imread(path, cv2.IMREAD_UNCHANGED) is not None]

# MediaPipe setup for hand detection
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=2, min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils

# Shape and score setup
color_scores = {"red": 0, "green": 0, "blue": 0, "image": 0}

# Game loop
shapes = []  # Store active shapes
last_shape_time = time.time()
shapes_dropped_count = 0
game_over = False

try: 
    while True:
        ret, frame = cap.read()
        if not ret:
            # Uncomment below if using video as the feed
            # if not game_over:
            #     cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            #     continue
            # else:
                break
    
        frame = cv2.flip(frame, 1) 
    
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb_frame)
        # Transpose the frame to fix the 90-degree rotation
        rgb_frame = rgb_frame.transpose((1, 0, 2))
        
        # Create a Pygame surface from the transposed video frame
        video_surface = pygame.surfarray.make_surface(rgb_frame)
    
        # Handle Pygame events (like closing the window or key presses for restarting)
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                game_over = True
                cap.release()
                break
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_q:
                    game_over = True
                    cap.release()
                    break
                elif event.key == pygame.K_r and game_over:
                    # Reset the game
                    game_over = False
                    shapes = []
                    shapes_dropped_count = 0
                    color_scores = {"red": 0, "green": 0, "blue": 0, 'image':0}
                    last_shape_time = time.time()
    
        if not game_over:
            # Add new shapes based on interval
            current_time = time.time()
            if current_time - last_shape_time > random.uniform(SHAPE_FALL_INTERVAL_MIN, SHAPE_FALL_INTERVAL_MAX):
                size = random.randint(20, 40)
                x = random.randint(size, SCREEN_WIDTH - size)
                y = 0
                vy = random.uniform(5, 20)
    
                if random.random() < IMAGE_FALL_PROBABILITY and loaded_images:
                    img_idx = random.randint(0, len(loaded_images) - 1)
                    shape = Shape(x, y, size, "image", "image", (255, 255, 255), vy, img_idx)
                else:
                    shape_type = random.choice(["circle", "square", "triangle"])
                    color, color_name = SHAPE_COLOR_SCHEME[shape_type]
                    shape = Shape(x, y, size, shape_type, color_name, color, vy)
                
                shapes.append(shape)
                last_shape_time = current_time
    
            # Update the screen
            screen.fill((0, 0, 0))  # Clear the screen
    
            # Draw the video frame to the background
            screen.blit(video_surface, (0, 0))
    
            # Update and draw all shapes
            updated_shapes = []
            for shape in shapes:
                if shape.update():
                    shape.draw(screen)
                    updated_shapes.append(shape)
                else:
                    shapes_dropped_count += 1
    
            shapes = updated_shapes
    
            if result.multi_hand_landmarks:
                for landmarks in result.multi_hand_landmarks:
                    mp_drawing.draw_landmarks(frame, landmarks)
                    h, w, _ = frame.shape
                    index_tip = landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP]
                    thumb_tip = landmarks.landmark[mp_hands.HandLandmark.THUMB_TIP]
                    index_x, index_y = int(index_tip.x * w), int(index_tip.y * h)
                    thumb_x, thumb_y = int(thumb_tip.x * w), int(thumb_tip.y * h)
    
                    cv2.circle(frame, (index_x, index_y), 5, (0, 255, 255), -1)
                    cv2.circle(frame, (thumb_x, thumb_y), 5, (255, 255, 0), -1)
    
                    # Calculate the distance between thumb and index finger tips
                    punch_distance = math.hypot(index_x - thumb_x, index_y - thumb_y)
                    if punch_distance < 40:  # Detect a "punch"
                        new_shapes = []
                        for shape in shapes:
                            x, y, size, shape_type, label, color, vy = shape
                            distance_to_finger = math.hypot(x - index_x, y - index_y)
                            if distance_to_finger < size + 10:  # Shape is "hit"
                                if shape_type != "image":
                                    color_scores[label] += 1
                                    print(f"{label.capitalize()} {shape_type} busted! Score: {color_scores[label]}")
                                else:
                                    color_scores[label] += 1
                                    print(f"Image object busted!")
                            else:
                                new_shapes.append(shape)
                        shapes = new_shapes
    
            # Display the score
            score_text = font.render(f"Red: {color_scores['red']} Green: {color_scores['green']} Blue: {color_scores['blue']} Image: {color_scores['image']}", True, (255, 255, 255))
            
            screen.blit(score_text, (10, SCREEN_HEIGHT - 50))
    
            # Check if game over
            if shapes_dropped_count >= MAX_SHAPES_DROPPED:
                game_over = True
    
        else:
            # Game over - stop updating game logic and show the game over screen
            display_game_over()
            # Uncomment if using video feed
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            
    
        pygame.display.update()
    
        # Control the frame rate (FPS)
        clock.tick(FPS)
except Exception as e:
    print(e)

finally:
    cap.release()
    pygame.quit()
    cv2.destroyAllWindows()

# Old

In [ ]:
import cv2
import random
import time
import numpy as np

In [ ]:
difficulty = int(input("Choose game difficulty between 1 & 10"))
HEIGHT = 768
WIDTH = 1366
RADIUS = 7
DIFFICULTY = range(1, 11)
DIFFICULTY = list(DIFFICULTY)

In [ ]:
def circlePosWidth(width, radius):
    randomLoc = random.randint(radius, width-radius)
    return randomLoc

In [ ]:
def drawMainCircles(canvas, main_x, main_y = 0, color = 255, thickness = -1):
    return cv2.circle(canvas, [main_x, main_y], RADIUS, color = color, thickness = thickness)

In [ ]:
i = 0
x = circlePosWidth(WIDTH, RADIUS)
while True:
    canvas = np.zeros([HEIGHT,WIDTH], dtype = np.uint8)
    cv2.circle(canvas, [x, i], RADIUS, 255, thickness = -1)
    i+=3
    if i > HEIGHT - RADIUS:
        i = 0
        x = circlePosWidth(WIDTH, RADIUS)
    cv2.imshow('canvas', canvas)
    cv2.waitKey(1)
    if cv2.waitKey(1) & 0xff == ord('q'):
        break
cv2.destroyAllWindows()


In [ ]:
cv2.imshow('canvas', canvas)
cv2.waitKey(0)
if cv2.waitKey(0) & 0xff == ord('q'):
    cv2.destroyAllWindows()

In [ ]:
def detectClick(event, x, y, flag, params):
    if event == cv2.EVENT_LBUTTONDOWN:
        # Loop through each spawned circle and check if the click is within the circle
        for spawned_circle in spawned_circles[:]:
            circle_x, circle_y = spawned_circle['x'], spawned_circle['y']
            # Check if the click is within the circle (simple distance check)
            if (x - circle_x)**2 + (y - circle_y)**2 <= RADIUS**2:
                print(f"Circle clicked: {spawned_circle}")
                # Remove the circle from the spawned_circles list
                spawned_circles.remove(spawned_circle)
                break  # Exit the loop once we remove the circle

In [ ]:
import cv2
import numpy as np
import random

HEIGHT, WIDTH = 768, 1366
RADIUS = 10
TARGET_FPS = 60
FRAME_TIME = 1 / TARGET_FPS
# Main circle (moves continuously down)
CIRCLES = {'main_circle': {'pos': [circlePosWidth(WIDTH, RADIUS), 0], 'timer': 0, 'interval': random.randint(10, 150)},
          'secondary_circle': {'pos': [circlePosWidth(WIDTH, RADIUS), 0], 'timer': 0, 'interval': random.randint(10, 150)},
          'thrid_circle': {'pos': [circlePosWidth(WIDTH, RADIUS), 0], 'timer': 0, 'interval': random.randint(10, 150)},
          'forth_circle': {'pos': [circlePosWidth(WIDTH, RADIUS), 0], 'timer': 0, 'interval': random.randint(10, 150)},
          'fifth_circle': {'pos': [circlePosWidth(WIDTH, RADIUS), 0], 'timer': 0, 'interval': random.randint(10, 150)}
          }

# Spawned Circle
spawned_circles = []

previous_time = time.time()

while True:
    canvas = np.zeros((HEIGHT, WIDTH), dtype=np.uint8)
    
    current_time = time.time()
    dt = current_time - previous_time
    previous_time = current_time
    
    for key in ['main_circle', 'secondary_circle', 'thrid_circle', 'forth_circle', 'fifth_circle']:
        circle = CIRCLES[key]
        circle_x, circle_y = circle['pos']

        circle['timer'] += 1
        if circle['timer'] >= circle['interval']:
            circle['timer'] = 0
            circle['interval'] = random.randint(600, 1200)
            spawned_circles.append({'x': circle_x, 'y': circle_y, 'interval': circle['interval'], 'color': random.randint(150, 255), 'speed': random.randint(1, 3)})
            print(key, circle['timer'], circle['interval'])

        time.sleep(2)

        for spawned_circle in spawned_circles[:]:
            if spawned_circle['y'] <= HEIGHT:
                # Draw the circle
                cv2.circle(canvas, (spawned_circle['x'], spawned_circle['y']), RADIUS, spawned_circle['color'], thickness=-1)
                
                # Move the circle down
                # movement_speed = (spawned_circle['speed'] * HEIGHT) / 480  # 480 is the reference canvas height

                # Move the circle down based on fixed movement speed (no frame delta scaling)
                speed = spawned_circle['speed'] * .5  # Apply the multiplier here

                # Move the circle down
                spawned_circle['y'] += 1
                # Temprary
                if spawned_circle['y'] >= 568:
                    spawned_circles.remove(spawned_circle)
                    

            else:
                # Remove the circle when it moves off the screen
                spawned_circles.remove(spawned_circle)

                
    cv2.line(canvas, [0, 578], [1366, 578], 255, 3, lineType=cv2.LINE_AA)
    cv2.imshow('canvas', canvas)
    # cv2.setMouseCallback('canvas', detectClick)
    if cv2.waitKey(20) & 0xff == ord('q'):
        break

cv2.destroyAllWindows()


In [ ]:
# # --- Main Circle: Always moving ---
    # drawMainCircles(canvas, main_x, main_y, color = 255, thickness = -1)
    # main_y += speed
    # if main_y > HEIGHT:
    #     main_y = 0
    #     speed = random.randint(3,5)
    #     main_x = circlePosWidth(WIDTH, RADIUS)
    #     CIRCLES['main_circle'] = [main_x, main_y]

    # # --- Secondary Circle: Random spawn ---
    # if secondary_active:
    #     # Move secondary circle
    #     cv2.circle(canvas, (secondary_x, secondary_y), RADIUS, 180, thickness=-1)
    #     secondary_y += 5
    #     if secondary_y > HEIGHT:
    #         secondary_active = False
    #         secondary_timer = random.randint(50, 150)
    # else:
    #     # Countdown to next spawn
    #     secondary_timer -= 1
    #     if secondary_timer <= 0:
    #         secondary_active = True
    #         secondary_x = random.randint(RADIUS, WIDTH - RADIUS)
    #         secondary_y = 0